# T6 · The check, and what it can say

**Tiny Transformer, Unit 2, module T6.** Every 500 steps, `train.py` stops to check the model on held-back text: the
same windows every time, no step taken. Here you score best.pt on those windows and on others, count what the check asks about,
split the final score by position, and replay how `train.py` picked the copy it kept.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T6-training-metrics.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T6-training-metrics.ipynb`.

**Computed here:** best.pt's scores on the check's windows and on 10 other draws, the check's asks counted, the loss by
position, and the keep-the-best rule replayed. **Checked against the run:** the file it saved, its log, and the course's numbers,
which the notebook asserts.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import hashlib, json, os, urllib.request
import torch

torch.set_num_threads(4)     # as the run did
print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole; then Unit 2's teaching functions, `code/training_steps.py`, copied in whole: T6's are the last four.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once but the first and the last few, which fill no
    window."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)


# ---------------------------------------------------------------------------------------------------- T6 · the check

def other_checks(model, ids, seeds, batches=100, B=64, T=32):
    """The check on other windows: for each seed, its own 100 batches of 64 starts, drawn as train.py draws the check's, scored
    as check_loss scores them. The model's numbers never change: only the windows do."""
    scores = []
    for seed in seeds:
        g = torch.Generator().manual_seed(seed)                   # another seed: other windows
        starts = [torch.randint(len(ids) - T - 1, (B,), generator=g) for _ in range(batches)]
        scores.append(check_loss(model, ids, starts, T))
    return scores                                                 # one score per seed


def how_often_asked(starts, n, T=32):
    """How many characters of a text the check's windows ask about 0 times, once, twice, …: times_asked over those windows,
    counted by times. The first and the last characters are never asked: no window starts before the first or reaches the
    last."""
    asked = times_asked(starts, n, T)                             # [n]   one count per character
    return torch.bincount(asked).tolist()                         # [how many 0 times, once, twice, …]


def loss_by_position(model, ids, T=32):
    """The final score, position by position: answer_costs reads the whole text in windows of 32, side by side; each
    position's costs are averaged down the windows. Position 0 has one character to go on, position 31 all 32."""
    p, _ = answer_costs(model, ids, T)                            # [n·T]  every right answer's probability, window after window
    return (-p.log()).view(-1, T).mean(0)                         # [T]    one average cost per position


def keep_best(checks, margin=1e-4, patience=20):
    """train.py's rule, replayed over a log of checks ([step, held-back score] pairs): a check that beats the best so far by
    more than `margin` keeps a copy, and the wait starts again; any other check adds 1 to the wait; a wait of `patience`
    would stop the run. Returns the kept steps, the step it would stop at (None: never), and the kept copy's score."""
    best, wait, kept = float("inf"), 0, []
    for step, score in checks:
        if score < best - margin:
            best, wait = score, 0                                 # a new best: keep a copy
            kept.append(step)
        elif step:
            wait += 1                                             # no new best: one more check waited
        if patience and wait >= patience:
            return kept, step, best                               # train.py would stop the run here
    return kept, None, best

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]                   # the first 90% trains; the rest is held back
_, val_starts = check_starts(len(train_ids), len(val_ids))  # the check's held-back windows: drawn once, the same every check

def find(*names):
    """A file the run left: in the labs' model folder, or in the course's results."""
    paths = [os.path.join(here, '..', 'model', names[0]),
             os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'results', 'prompter-r1', names[1])]
    found = [p for p in paths if os.path.exists(p)]
    assert found, f"{names[0]} is not here: run the Setup cell at the top"
    return found[0]
best = Prompter(cfg)
best.load_state_dict(torch.load(find('prompter-r1-best.pt', 'best.pt'), weights_only=True))   # the numbers train.py kept
best.eval()
log = json.load(open(find('prompter-r1-train.json', 'train.json')))['log']                   # the run's 201 checks
print(f"{len(val_ids):,} held-back characters · {len(val_starts)} batches of {len(val_starts[0])} windows · {len(log)} checks in the log")
assert len(val_ids) == 111540 and len(log) == 201

111,540 held-back characters · 100 batches of 64 windows · 201 checks in the log


# Section 1 · What the check reads

What the check reads every 500 steps, how far its score can move on other windows, and how much of the text it asks about. Each chapter below
follows its page: Picture it, Watch it in our run, then Read and run the code.

## T6.1 · The same windows every time

*Why does every check read the same windows?*

**Predict first** (the page asks it too, before its clip): every check reads the same 6,400 windows of held-back text, drawn once with their own seed. Score best.pt on 100
other draws of 6,400 windows, its numbers unchanged. How far apart are the highest and lowest scores: about 0.02,
small but not nothing; about 0.002, as 6,400 windows are plenty; or about 0.2, as each draw is a different test?

### T6.1.1 · Picture it

Two checks should differ only because the numbers changed. How can a check make sure? Picture a made-up quiz of 10 random words,
for a learner who knows 7 in 10. Three quizzes on other words score 6, 8 and 7. The learner never changed: only the words did. The
same 10 words every time score the same: 8, a bit lucky, but always 8. Any change is the learner's. (The page's quiz is made up to
show the idea.)

### T6.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is best.pt, scored on 100 other draws of 6,400 held-back windows. Its numbers never change. The scores run
from 1.922 to 1.945: 0.023 apart. The run's best check so far improved by just
0.004, from step 80,000 to step 98,000. On other windows each time, a gain that small could hide inside that
spread. The same windows let two checks differ only because the numbers did.

### T6.1.3 · Read and run the code

`other_checks` in `code/training_steps.py` (copied in above) scores a model on other windows. Each seed gives its own generator,
and its own 100 batches of 64 windows; `check_loss` scores each draw just as the run's check scores its own. The cell first scores
best.pt on the check's own windows (1.9351, the run's best check), then on seeds 1 to 10: the first 10 of
the page's 100 draws. (About a minute.)

In [7]:
own = check_loss(best, val_ids, val_starts)
print(f"the check's own windows: {own:.4f} (the run's best check, step 98,000: 1.9351)")
assert abs(own - 1.9350941205024719) < 1e-4

others = other_checks(best, val_ids, range(1, 11))
for seed, score in zip(range(1, 11), others):
    print(f"seed {seed:>2}: {score:.4f}")
print(f"{min(others):.3f} to {max(others):.3f}: {max(others) - min(others):.3f} apart, on 10 draws (on the page's 100: 0.023)")
assert all(abs(a - b) < 1e-4 for a, b in zip(others, [1.927098, 1.940547, 1.927882, 1.934833, 1.927796, 1.93505, 1.927997, 1.935604, 1.926547, 1.944608]))

the check's own windows: 1.9351 (the run's best check, step 98,000: 1.9351)


seed  1: 1.9271
seed  2: 1.9405
seed  3: 1.9279
seed  4: 1.9348
seed  5: 1.9278
seed  6: 1.9350
seed  7: 1.9280
seed  8: 1.9356
seed  9: 1.9265
seed 10: 1.9446
1.927 to 1.945: 0.018 apart, on 10 draws (on the page's 100: 0.023)


### ✋ Your turn

How much did the run's best check so far improve from step 80,000 to step 98,000? Find both in `log` (each entry has a `step` and
a `val`): the best `val` up to each step. Put the difference in `answer`.

In [8]:
best_by = lambda s: min(e['val'] for e in log if e['step'] <= s)
answer = None   # TODO: best_by(80000) - best_by(98000)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (best_by(80000) - best_by(98000))) < 1e-9, 'the best up to each step: best_by(80000) - best_by(98000)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
best_by = lambda s: min(e['val'] for e in log if e['step'] <= s)
answer = best_by(80000) - best_by(98000)
print(f'the late gain: {answer:.4f}, less than the {max(others) - min(others):.3f} spread of the other windows')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (best_by(80000) - best_by(98000))) < 1e-9, 'the best up to each step: best_by(80000) - best_by(98000)'
    print("✓ right:", answer)

the late gain: 0.0039, less than the 0.018 spread of the other windows
✓ right: 0.0039199999999999235


## T6.2 · Two scores for the same numbers

*Does the check read the whole held-back text?*

**Predict first** (the page asks it too, before its clip): the check reads 6,400 windows of 32: it asks 204,800 times. The held-back text has
111,540 characters. How much of it does the check ask about: about 6 in
7 characters, some many times; nearly every character, about twice each; or every
character once, then some again?

### T6.2.1 · Picture it

How much of the held-back text does the check read? Picture a made-up text of 12 characters, and one window of 4, starting at
place 3. The window asks about the 4 characters after its start: 4 asks. With 3 windows like it: 12 asks, as many asks as the text
has characters. (The page's text is made up to show the idea.)

### T6.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the held-back text, its 111,540 characters counted by how often the check asks about each. Of them,
17,008 are never asked. Others are asked again and again, up to 10 times. At the end, `train.py` also
scored best.pt on the whole held-back text, nearly every character once. It scored 1.932, against the check's 1.935: close, but
not the same reading.

### T6.2.3 · Read and run the code

`how_often_asked` counts the check's asks. `times_asked` counts, for every character, the windows that ask about it (the 32
characters after every window's start); the counts, counted by times, add up to every held-back character, and the asks to 204,800. Then `answer_costs` reads
the whole held-back text in windows side by side, as `train.py`'s final score does: 111,520 answers, every character
once but the first and the last few, which fill no window.

In [10]:
counts = how_often_asked(val_starts, len(val_ids))
for times, n in enumerate(counts):
    print(f"asked {times:>2} times: {n:>6,} characters")
print(f"{sum(counts):,} characters · {sum(t * n for t, n in enumerate(counts)):,} asks")
assert counts == [17008, 33517, 29819, 18601, 8579, 2915, 836, 202, 49, 11, 3]

p, _ = answer_costs(best, val_ids)
final = (-p.log()).mean().item()
print(f"the whole held-back text, {len(p):,} answers: {final:.4f} (train.py's final score: 1.9317) · the check: {own:.4f}")
assert len(p) == 111520 and abs(final - 1.9317) < 1e-4

asked  0 times: 17,008 characters
asked  1 times: 33,517 characters
asked  2 times: 29,819 characters
asked  3 times: 18,601 characters
asked  4 times:  8,579 characters
asked  5 times:  2,915 characters
asked  6 times:    836 characters
asked  7 times:    202 characters
asked  8 times:     49 characters
asked  9 times:     11 characters
asked 10 times:      3 characters
111,540 characters · 204,800 asks
the whole held-back text, 111,520 answers: 1.9317 (train.py's final score: 1.9317) · the check: 1.9351


### ✋ Your turn

What share of the held-back characters does the check never ask about? Divide `counts[0]` by the number of held-back characters,
and put it in `answer`.

In [11]:
answer = None   # TODO: counts[0] / len(val_ids)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - counts[0] / len(val_ids)) < 1e-12, 'never asked, over all of them: counts[0] / len(val_ids)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = counts[0] / len(val_ids)
print(f'{answer:.1%} never asked: about 1 in {round(1 / answer)}')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - counts[0] / len(val_ids)) < 1e-12, 'never asked, over all of them: counts[0] / len(val_ids)'
    print("✓ right:", answer)

15.2% never asked: about 1 in 7
✓ right: 0.15248341402187557


# Section 2 · Reading the scores

The check's scores read more closely: the loss at each position of a window, and the rule that picked the copy the run kept.

## T6.3 · Position by position

*Does every position of a window predict as well?*

**Predict first** (the page asks it too, before its clip): in a window, position 0 has 1 character to go on; position 31 has all 32. On the held-back text, how does best.pt's loss
change from position 0 to 31: it falls over the first few positions, then stays about the same; it keeps falling just as
fast, all the way to position 31; or it hardly changes, as one character is enough to guess the next?

### T6.3.1 · Picture it

Each position of a window predicts the next character. What can each one read? Picture a made-up window of 8 characters: to be or.
Position 0 reads one character. Position 7 reads all 8. Each position reads only itself and what is to its left: further along,
more to go on. (The page's window is made up to show the idea.)

### T6.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is best.pt's loss at each of the 32 positions, over the whole held-back text. Position 0 has one character to go on:
2.59. Position 1: 2.20. Position 2: 2.00. From position 4 on, it stays between 1.83 and
1.94, all the way to position 31. A few characters back help a lot. For our tiny model, more of the window
adds little.

### T6.3.3 · Read and run the code

`loss_by_position` splits the final score by position. `answer_costs` reads the whole text in windows of 32, side by side; each
position's costs are averaged down the windows: 2.59 at position 0, about 1.9 from position 4. All 32 averaged together
give the final score again: 1.932.

In [13]:
by_pos = loss_by_position(best, val_ids)
for i, v in enumerate(by_pos.tolist()):
    print(f"position {i:>2}: {v:.3f}  " + "#" * round(40 * (v - 1.7)))
print(f"all 32 averaged: {by_pos.mean().item():.4f} (the final score: 1.9317)")
assert all(abs(a - b) < 1e-4 for a, b in zip(by_pos.tolist(), [2.589739, 2.19726, 1.998129, 1.964595, 1.934955, 1.923672, 1.894875, 1.915397, 1.87863, 1.915747, 1.936882, 1.879574, 1.884778, 1.921861, 1.932473, 1.929443, 1.880055, 1.931371, 1.887155, 1.910435, 1.872114, 1.896867, 1.856734, 1.866808, 1.911867, 1.870755, 1.843314, 1.918321, 1.857607, 1.831351, 1.875174, 1.906354]))
assert abs(by_pos.mean().item() - 1.9317) < 1e-4

position  0: 2.590  ####################################
position  1: 2.197  ####################
position  2: 1.998  ############
position  3: 1.965  ###########
position  4: 1.935  #########
position  5: 1.924  #########
position  6: 1.895  ########
position  7: 1.915  #########
position  8: 1.879  #######
position  9: 1.916  #########
position 10: 1.937  #########
position 11: 1.880  #######
position 12: 1.885  #######
position 13: 1.922  #########
position 14: 1.932  #########
position 15: 1.929  #########
position 16: 1.880  #######
position 17: 1.931  #########
position 18: 1.887  #######
position 19: 1.910  ########
position 20: 1.872  #######
position 21: 1.897  ########
position 22: 1.857  ######
position 23: 1.867  #######
position 24: 1.912  ########
position 25: 1.871  #######
position 26: 1.843  ######
position 27: 1.918  #########
position 28: 1.858  ######
position 29: 1.831  #####
position 30: 1.875  #######
position 31: 1.906  ########
all 32 averaged: 1.9317 (the fina

### ✋ Your turn

How much does the loss fall from position 0 to position 4, and how much from position 4 to the last? Put the two falls, as a pair,
in `answer`.

In [14]:
answer = None   # TODO: (by_pos[0] - by_pos[4]).item(), (by_pos[4] - by_pos[-1]).item()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer[0] - (by_pos[0] - by_pos[4]).item()) < 1e-9 and abs(answer[1] - (by_pos[4] - by_pos[-1]).item()) < 1e-9, 'two differences: by_pos[0] - by_pos[4], then by_pos[4] - by_pos[-1]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = (by_pos[0] - by_pos[4]).item(), (by_pos[4] - by_pos[-1]).item()
print(f'positions 0 to 4: {answer[0]:.2f} · 4 to the last: {answer[1]:.2f}')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer[0] - (by_pos[0] - by_pos[4]).item()) < 1e-9 and abs(answer[1] - (by_pos[4] - by_pos[-1]).item()) < 1e-9, 'two differences: by_pos[0] - by_pos[4], then by_pos[4] - by_pos[-1]'
    print("✓ right:", answer)

positions 0 to 4: 0.65 · 4 to the last: 0.03
✓ right: (0.654784083366394, 0.028600573539733887)


## T6.4 · Keeping the best

*Which numbers does the run keep?*

**Predict first** (the page asks it too, before its clip): at each of the run's 201 checks, `train.py` keeps a copy of the numbers when the held-back score beats the best so far by
more than 0.0001. How many times did it keep one: about 40, often at first, then
seldom; about 190, as nearly every check beat the best so far; or about
15, as after the fast start new bests almost stopped?

### T6.4.1 · Picture it

Which of its numbers should a run keep? Its training text alone cannot say. Picture a made-up run. On its training text, the score
keeps falling. On held-back text, it falls, then rises: the model is memorising. It keeps getting better on its training text,
and worse on new text. So a run keeps a copy whenever
the held-back score is its best so far. (The page's run is made up to show the idea.)

### T6.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's 201 held-back checks, with every copy it kept marked. It kept a copy 43 times, step 0
included: 15 in the first 10,000 steps. The longest wait was 17 checks, from step
47,000 to 56,000. A wait of 20 would have stopped the run. Each copy replaced the one before. The
last, at step 98,000, scored 1.935: best.pt.

### T6.4.3 · Read and run the code

`keep_best` replays `train.py`'s rule over a log of checks. A check that beats the best by more than the margin keeps a copy, and
the wait starts again; any other check adds 1 to the wait. A wait as long as the patience, 20, stops the run. On the run's log:
43 copies, and it never stopped. With
patience 10, it would stop at step 24,500; its copy would then come from step 19,500: 1.977, not
1.935.

In [16]:
checks = [(e['step'], e['val']) for e in log]
kept, stop, score = keep_best(checks)
print(f"margin 0.0001, patience 20: {len(kept)} copies, the last at step {kept[-1]:,} ({score:.4f}) · stopped: {stop}")
print("the copies' steps:", kept)
assert kept == [0, 500, 1000, 1500, 2000, 2500, 3500, 4000, 4500, 5000, 5500, 6000, 7000, 8000, 8500, 12500, 13000, 16000, 18000, 19500, 26500, 27000, 32000, 34000, 42000, 42500, 43500, 47000, 56000, 60500, 61500, 62000, 65000, 71500, 74000, 78000, 79000, 84000, 86000, 90500, 91000, 92500, 98000] and stop is None

for patience in (5, 10, 20):
    kept_p, stop_p, score_p = keep_best(checks, patience=patience)
    print(f"patience {patience:>2}: stops at step {stop_p if stop_p is None else f'{stop_p:,}'} · keeps step {kept_p[-1]:,}, {score_p:.4f}")
    assert [stop_p, kept_p[-1]] == [[11000, 8500], [24500, 19500], [None, 98000]][(5, 10, 20).index(patience)]

margin 0.0001, patience 20: 43 copies, the last at step 98,000 (1.9351) · stopped: None
the copies' steps: [0, 500, 1000, 1500, 2000, 2500, 3500, 4000, 4500, 5000, 5500, 6000, 7000, 8000, 8500, 12500, 13000, 16000, 18000, 19500, 26500, 27000, 32000, 34000, 42000, 42500, 43500, 47000, 56000, 60500, 61500, 62000, 65000, 71500, 74000, 78000, 79000, 84000, 86000, 90500, 91000, 92500, 98000]
patience  5: stops at step 11,000 · keeps step 8,500, 1.9968
patience 10: stops at step 24,500 · keeps step 19,500, 1.9773
patience 20: stops at step None · keeps step 98,000, 1.9351


### ✋ Your turn

With a margin of 0.01 instead of 0.0001, a check must beat the best by far more. How many copies would the run keep (no
stopping: `patience=0`)? Put the number in `answer`.

In [17]:
answer = None   # TODO: len(keep_best(checks, margin=0.01, patience=0)[0])

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 13, "the kept steps' count: len(keep_best(checks, margin=0.01, patience=0)[0])"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
answer = len(keep_best(checks, margin=0.01, patience=0)[0])
print(f'margin 0.01: {answer} copies')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 13, "the kept steps' count: len(keep_best(checks, margin=0.01, patience=0)[0])"
    print("✓ right:", answer)

margin 0.01: 13 copies
✓ right: 13


Every check reads the same 6,400 held-back windows: on other windows, best.pt's score spread 0.023,
more than the run's best check gained late in the run, 0.004. Those windows leave 17,008 held-back
characters unasked; at the end, `train.py` also reads nearly the whole held-back text once: 1.932.
Split by position, the loss falls from 2.59 at position 0 to about 1.9 by position 4, then stays there. And `train.py` kept
a new copy 43 times, each replacing the one before; the last, from step 98,000, is best.pt.

Next: **T7 · the curve**, the 201 checks drawn as one line.